# Evaluation 2

In [1]:
import pandas as pd
import itertools

In [2]:
df_evaluation = pd.read_csv('./evaluation.csv')

In [3]:
df_evaluation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 94 columns):
 #   Column                                                                           Non-Null Count  Dtype  
---  ------                                                                           --------------  -----  
 0   document                                                                         30 non-null     object 
 1   question_type                                                                    30 non-null     object 
 2   question                                                                         30 non-null     object 
 3   reference_answer                                                                 30 non-null     object 
 4   mistral:SentenceSplitter(chunk_size=128, chunk_overlap=10):answer                30 non-null     object 
 5   mistral:SentenceSplitter(chunk_size=128, chunk_overlap=10):contexts              30 non-null     object 
 6   mistral:Sent

In [4]:
strategy_labels = {
    "SentenceSplitter(chunk_size=128, chunk_overlap=10)": "SS-128",
    "SentenceSplitter(chunk_size=256, chunk_overlap=10)": "SS-256",
    "SentenceSplitter(chunk_size=512, chunk_overlap=10)": "SS-512",
    "SemanticSplitterNodeParser": "Semantic",
    "SentenceWindowNodeParser": "SentWindow"
}

## Performance

In [5]:
configs = {
    "llms": ["mistral", "llama"],
    "chunk_strategies": [
        "SentenceSplitter(chunk_size=128, chunk_overlap=10)",
        "SentenceSplitter(chunk_size=256, chunk_overlap=10)",
        "SentenceSplitter(chunk_size=512, chunk_overlap=10)",
        "SemanticSplitterNodeParser",
        "SentenceWindowNodeParser"
    ],
    "eval_metrics": ['faithfulness', 'answer_relevancy', 'nv_context_relevance', 'task_completion']
}
total_configs = list(itertools.product(*configs.values()))
total_configs = [f'{llm}:{chunk_strat}:{metrics}' for llm, chunk_strat, metrics in total_configs]
df_means = df_evaluation[total_configs].mean()
llm_chunk_configs = list(itertools.product(configs['llms'], configs['chunk_strategies']))
df_perf = pd.DataFrame({
    "llm": [llm for llm, chunk_strat in llm_chunk_configs],
    "chunking_strategy": [strategy_labels[chunk_strat] for llm, chunk_strat in llm_chunk_configs],
    "faithfulness": df_means[[f'{llm}:{chunk_strat}:faithfulness' for llm, chunk_strat in llm_chunk_configs]].to_list(),
    "answer_relevancy": df_means[[f'{llm}:{chunk_strat}:answer_relevancy' for llm, chunk_strat in llm_chunk_configs]].to_list(),
    "context_relevance": df_means[[f'{llm}:{chunk_strat}:nv_context_relevance' for llm, chunk_strat in llm_chunk_configs]].to_list(),
    "task_completion": df_means[[f'{llm}:{chunk_strat}:task_completion' for llm, chunk_strat in llm_chunk_configs]].to_list(),
})
df_perf.loc[df_perf['llm'].duplicated(), 'llm'] = ''
df_perf.loc[df_perf['llm']=='mistral', 'llm'] = 'Mistral-7B'
df_perf.loc[df_perf['llm']=='llama', 'llm'] = 'Llama-3.2-3B'
print(df_perf.to_markdown(index=False, floatfmt=".2f"))

| llm          | chunking_strategy   |   faithfulness |   answer_relevancy |   context_relevance |   task_completion |
|:-------------|:--------------------|---------------:|-------------------:|--------------------:|------------------:|
| Mistral-7B   | SS-128              |           0.83 |               0.58 |                0.84 |              0.73 |
|              | SS-256              |           0.88 |               0.64 |                0.82 |              0.77 |
|              | SS-512              |           0.92 |               0.64 |                0.78 |              0.83 |
|              | Semantic            |           0.85 |               0.48 |                0.53 |              0.53 |
|              | SentWindow          |           0.87 |               0.60 |                0.79 |              0.80 |
| Llama-3.2-3B | SS-128              |           0.87 |               0.56 |                0.84 |              0.73 |
|              | SS-256              |          

## Memory

In [6]:
configs = {
    "chunk_strategies": [
        "SentenceSplitter(chunk_size=128, chunk_overlap=10)",
        "SentenceSplitter(chunk_size=256, chunk_overlap=10)",
        "SentenceSplitter(chunk_size=512, chunk_overlap=10)",
        "SemanticSplitterNodeParser",
        "SentenceWindowNodeParser"
    ],
    "eval_metrics": ['total_token', 'num_chunk', 'mem_usage']
}

strategy_labels = {
    "SentenceSplitter(chunk_size=128, chunk_overlap=10)": "SS-128",
    "SentenceSplitter(chunk_size=256, chunk_overlap=10)": "SS-256",
    "SentenceSplitter(chunk_size=512, chunk_overlap=10)": "SS-512",
    "SemanticSplitterNodeParser": "Semantic",
    "SentenceWindowNodeParser": "SentWindow"
}

llms = ["mistral", "llama"]
df_eval_union = []
for llm in llms:
    marignal_cols = [f'{llm}:{chunk_strat}:{metrics}' for chunk_strat, metrics in itertools.product(*configs.values())]
    df_eval_marginal = df_evaluation[marignal_cols]
    df_eval_marginal.columns = [f'{chunk_strat}:{metrics}' for chunk_strat, metrics in itertools.product(*configs.values())]
    df_eval_union.append(df_eval_marginal)
df_eval_union = pd.concat(df_eval_union, axis=0)

total_configs = list(itertools.product(*configs.values()))
total_configs = [f'{chunk_strat}:{metrics}' for chunk_strat, metrics in total_configs]
df_means = df_eval_union[total_configs].mean()
df_mins = df_eval_union[total_configs].min()
df_maxs = df_eval_union[total_configs].max()

df_mem = pd.DataFrame({
    "Chunking Strategy": [strategy_labels[s] for s in configs['chunk_strategies']],
    "Total Tokens (min / mean / max)": [
        f"{int(mn)} / {mean:.2f} / {int(mx)}"
        for mn, mean, mx in zip(
            df_mins[[f'{s}:total_token' for s in configs['chunk_strategies']]],
            df_means[[f'{s}:total_token' for s in configs['chunk_strategies']]],
            df_maxs[[f'{s}:total_token' for s in configs['chunk_strategies']]]
        )
    ],
    "Num Chunks (min / mean / max)": [
        f"{int(mn)} / {mean:.2f} / {int(mx)}"
        for mn, mean, mx in zip(
            df_mins[[f'{s}:num_chunk' for s in configs['chunk_strategies']]],
            df_means[[f'{s}:num_chunk' for s in configs['chunk_strategies']]],
            df_maxs[[f'{s}:num_chunk' for s in configs['chunk_strategies']]]
        )
    ],
    "Mem Usage in KB (min / mean / max)": [
        f"{mn:.0f} / {mean:.2f} / {mx:.0f}"
        for mn, mean, mx in zip(
            df_mins[[f'{s}:mem_usage' for s in configs['chunk_strategies']]].to_numpy() / 1024,
            df_means[[f'{s}:mem_usage' for s in configs['chunk_strategies']]].to_numpy() / 1024,
            df_maxs[[f'{s}:mem_usage' for s in configs['chunk_strategies']]].to_numpy() / 1024
        )
    ],
})
print(df_mem.to_markdown(index=False))

| Chunking Strategy   | Total Tokens (min / mean / max)   | Num Chunks (min / mean / max)   | Mem Usage in KB (min / mean / max)   |
|:--------------------|:----------------------------------|:--------------------------------|:-------------------------------------|
| SS-128              | 1403 / 1448.23 / 1497             | 13 / 14.63 / 16                 | 446 / 446.77 / 451                   |
| SS-256              | 1291 / 1387.17 / 1482             | 6 / 6.43 / 8                    | 253 / 253.85 / 256                   |
| SS-512              | 1135 / 1338.87 / 1454             | 3 / 3.00 / 3                    | 156 / 179.12 / 194                   |
| Semantic            | 0 / 680.60 / 1441                 | 0 / 1.47 / 4                    | 150 / 181.75 / 213                   |
| SentWindow          | 804 / 1296.47 / 1498              | 2 / 19.93 / 46                  | 880 / 880.59 / 884                   |
